# Caso principal — CallMeMaybe: identificación de operadores ineficaces

## Entregable 1: Descomposición de tareas

Este documento define **qué se va a analizar, cómo y por qué**, antes de escribir el análisis. Su función es dejar explícitos los criterios y las decisiones metodológicas para que puedan discutirse y corregirse antes de invertir en la implementación.

El análisis propiamente dicho se desarrolla en el notebook `02_analisis_operadores.ipynb`.

---

**Contenido**

1. Contexto de negocio
2. Problema y objetivo del estudio
3. Diagnóstico preliminar de los datos
4. Definición operativa de "operador ineficaz"
5. Preguntas de investigación
6. Hipótesis a contrastar
7. Plan de limpieza y decisiones justificadas
8. Etapas del análisis
9. Entregables y riesgos

---
## 1. Contexto de negocio

CallMeMaybe es un servicio de telefonía virtual. Sus clientes son **organizaciones**, no personas: empresas que necesitan repartir un gran volumen de llamadas entrantes entre varios operadores, o realizar llamadas salientes a través de ellos. Los operadores también se comunican entre sí mediante llamadas internas dentro de la red del servicio.

La empresa está desarrollando una función nueva para que **los supervisores de cada organización cliente** puedan detectar a sus operadores menos eficaces. El valor de esa función no está en producir un ranking, sino en permitir una intervención concreta: redistribuir carga, reforzar formación o revisar la dotación de personal.

Esto condiciona el análisis desde el inicio: el resultado debe ser **accionable para un supervisor**, que no es un perfil técnico y que necesita saber a quién mirar y por qué motivo.

---
## 2. Problema y objetivo del estudio

El enunciado de negocio describe a un operador ineficaz con tres señales:

> Un operador es ineficaz si tiene una gran cantidad de llamadas entrantes perdidas (internas y externas) y un tiempo de espera prolongado para las llamadas entrantes. Además, si se supone que un operador debe realizar llamadas salientes, un número reducido de ellas también será un signo de ineficacia.

Ese enunciado no es directamente medible: "gran cantidad", "prolongado" y "reducido" no tienen umbral, y la tercera señal viene con una condición explícita que hay que respetar. **La primera tarea es convertirlo en criterios calculables y defendibles** — pero antes hay que comprobar si los datos permiten medir las tres señales, que es el objeto de la sección 3.

### Objetivo general

Identificar, con criterios explícitos y reproducibles, qué operadores presentan un desempeño deficiente, y entregar a los supervisores un diagnóstico accionable.

### Objetivos específicos

1. Realizar el análisis exploratorio de los datos y evaluar su calidad.
2. Construir una definición operativa de ineficacia y aplicarla para identificar a los operadores afectados.
3. Contrastar estadísticamente hipótesis relevantes sobre el desempeño del servicio.
4. Traducir los hallazgos en recomendaciones concretas.

---
## 3. Diagnóstico preliminar de los datos

Definir criterios sin haber mirado los datos lleva a planes que no se pueden ejecutar. Esta sección los inspecciona sin modificarlos, para fundamentar las decisiones de las secciones siguientes.

In [ ]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

RUTA_DATOS = 'data/raw/main_project/'

dataset = pd.read_csv(RUTA_DATOS + 'telecom_dataset_new.csv')
clientes = pd.read_csv(RUTA_DATOS + 'telecom_clients.csv')

print(f'telecom_dataset_new.csv : {dataset.shape[0]:>6,} filas x {dataset.shape[1]} columnas')
print(f'telecom_clients.csv     : {clientes.shape[0]:>6,} filas x {clientes.shape[1]} columnas')

telecom_dataset_new.csv : 53,902 filas x 9 columnas
telecom_clients.csv     :    732 filas x 3 columnas


> **Nota sobre los nombres de archivo.** Las instrucciones se refieren a `telecom_dataset_us.csv` y `telecom_clients_us.csv`. Los archivos disponibles son `telecom_dataset_new.csv` y `telecom_clients.csv`.

In [2]:
dataset.head()

,user_id,date,direction,internal,operator_id,is_missed_call,calls_count,call_duration,total_call_duration
0,166377,2019-08-04 00:00:00+03:00,in,False,NaN,True,2,0,4
1,166377,2019-08-05 00:00:00+03:00,out,True,880022.0,True,3,0,5
2,166377,2019-08-05 00:00:00+03:00,out,True,880020.0,True,1,0,1
3,166377,2019-08-05 00:00:00+03:00,out,True,880020.0,False,1,10,18
4,166377,2019-08-05 00:00:00+03:00,out,False,880022.0,True,3,0,25


In [3]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53902 entries, 0 to 53901
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   user_id              53902 non-null  int64  
 1   date                 53902 non-null  object 
 2   direction            53902 non-null  object 
 3   internal             53785 non-null  object 
 4   operator_id          45730 non-null  float64
 5   is_missed_call       53902 non-null  bool   
 6   calls_count          53902 non-null  int64  
 7   call_duration        53902 non-null  int64  
 8   total_call_duration  53902 non-null  int64  
dtypes: bool(1), float64(1), int64(4), object(3)
memory usage: 3.3+ MB


In [4]:
clientes.head()

,user_id,tariff_plan,date_start
0,166713,A,2019-08-15
1,166901,A,2019-08-23
2,168527,A,2019-10-29
3,167097,A,2019-09-01
4,168193,A,2019-10-16


### 3.1 Calidad de los datos

In [5]:
diagnostico = pd.DataFrame({
    'tipo': dataset.dtypes.astype(str),
    'nulos': dataset.isna().sum(),
    '% nulos': (dataset.isna().mean() * 100).round(2),
    'valores_unicos': dataset.nunique(),
})
diagnostico

,tipo,nulos,% nulos,valores_unicos
user_id,int64,0,0.00,307
date,object,0,0.00,119
direction,object,0,0.00,2
internal,object,117,0.22,2
operator_id,float64,8172,15.16,1092
is_missed_call,bool,0,0.00,2
calls_count,int64,0,0.00,502
call_duration,int64,0,0.00,5373
total_call_duration,int64,0,0.00,6040


In [ ]:
duplicados = dataset.duplicated().sum()
fechas = pd.to_datetime(dataset['date'], utc=True)

print(f'Duplicados exactos          : {duplicados:,} ({duplicados / len(dataset):.1%} de las filas)')
print(f'Período cubierto            : {fechas.min().date()} a {fechas.max().date()}')
print(f'Clientes con actividad     : {dataset["user_id"].nunique()} de {clientes["user_id"].nunique()} registrados')
print(f'Operadores identificados    : {dataset["operator_id"].nunique():,}')
print()

espera = dataset['total_call_duration'] - dataset['call_duration']
print(f'Esperas negativas           : {(espera < 0).sum()}  (0 = la derivación del tiempo de espera es coherente)')
print(f'Duración 0 sin ser perdida  : {((dataset["call_duration"] == 0) & (~dataset["is_missed_call"])).sum()}')

Duplicados exactos          : 4,900 (9.1% de las filas)
Período cubierto            : 2019-08-01 a 2019-11-27
Clientes con actividad      : 307 de 732 registrados
Operadores identificados    : 1,092

Esperas negativas           : 0  (0 = la derivación del tiempo de espera es coherente)
Duración 0 sin ser perdida  : 20


### 3.2 ¿Son los duplicados un error o registros legítimos?

El dataset es una **agregación diaria**, no un registro por llamada. Su clave lógica es la combinación cliente + fecha + dirección + interna + operador + perdida. Si dos filas comparten esa clave pero tienen métricas distintas, se trata de bloques de agregación diferentes y hay que conservarlas; si son idénticas byte a byte, se trata de un registro escrito dos veces.


In [7]:
clave_logica = ['user_id', 'date', 'direction', 'internal', 'operator_id', 'is_missed_call']

dup_exactos = dataset.duplicated().sum()
dup_por_clave = dataset.duplicated(subset=clave_logica).sum()

print(f'Duplicados exactos (todas las columnas)     : {dup_exactos:,}')
print(f'Duplicados por clave lógica                 : {dup_por_clave:,}')
print(f'Repeticiones de clave con métricas distintas: {dup_por_clave - dup_exactos:,}')
print()

copias = dataset[dataset.duplicated(keep=False)].groupby(clave_logica, dropna=False).size()
print(f'Número de copias por grupo duplicado: {sorted(int(c) for c in copias.unique())}')
print()

mes = fechas.dt.tz_convert(None).dt.to_period('M')
por_mes = (dataset[dataset.duplicated(keep=False)].assign(mes=mes).groupby('mes').size()
           / dataset.assign(mes=mes).groupby('mes').size())
print('Proporción de filas duplicadas por mes:')
print(por_mes.round(3).to_string())

Duplicados exactos (todas las columnas)     : 4,900
Duplicados por clave lógica                 : 4,900
Repeticiones de clave con métricas distintas: 0

Número de copias por grupo duplicado: [2]

Proporción de filas duplicadas por mes:
mes
2019-08    0.176
2019-09    0.183
2019-10    0.185
2019-11    0.179
Freq: M


**Conclusión:** ninguna clave lógica se repite con métricas distintas, todos los grupos duplicados tienen exactamente dos copias idénticas, y la proporción es constante a lo largo de los cuatro meses (~18%). Un fenómeno uniforme en el tiempo y sin variación en los valores no corresponde a actividad real, sino a un **fallo sistemático del proceso de carga**.

### 3.3 ¿Se pueden atribuir las llamadas entrantes perdidas a un operador?

La primera señal de ineficacia del enunciado son las llamadas entrantes perdidas, y el 15% de las filas no tiene `operator_id`. Antes de construir cualquier métrica hay que saber dónde se concentran esos nulos.

In [ ]:
sin_dup = dataset.drop_duplicates()
entrantes = sin_dup[sin_dup['direction'] == 'in']

resumen = (entrantes
           .assign(operador=lambda x: x['operator_id'].isna().map({True: 'sin operador', False: 'con operador'}))
           .groupby(['operador', 'is_missed_call'])['calls_count'].sum()
           .unstack(fill_value=0))
resumen.columns = ['contestadas', 'perdidas']
display(resumen)

perdidas = entrantes[entrantes['is_missed_call']]
sin_op = perdidas['operator_id'].isna()

print(f'Llamadas entrantes perdidas totales        : {perdidas["calls_count"].sum():,}')
print(f'  ... sin operador asignado                : {perdidas.loc[sin_op, "calls_count"].sum():,} '
      f'({perdidas.loc[sin_op, "calls_count"].sum() / perdidas["calls_count"].sum():.1%})')
print(f'  ... atribuibles a un operador concreto   : {perdidas.loc[~sin_op, "calls_count"].sum():,} '
      f'repartidas entre {perdidas.loc[~sin_op, "operator_id"].nunique()} operadores')
print()
print(f'Tasa global de pérdida en llamadas entrantes: '
      f'{perdidas["calls_count"].sum() / entrantes["calls_count"].sum():.1%}')
print(f'Clientes afectados por pérdidas sin operador: {perdidas.loc[sin_op, "user_id"].nunique()} de {sin_dup["user_id"].nunique()}')

NameError: name 'dataset' is not defined

: 

**Hallazgo:** El 99% de las llamadas entrantes perdidas no tiene operador asignado, y afecta prácticamente a todos los clientes. El resultado es coherente con la naturaleza del fenómeno: una llamada entrante perdida es, por definición, una llamada que *ningún* operador atendió, de modo que no hay a quién atribuirla.

Sea porque el sistema no registra el operador en ese caso, o porque realmente no hubo ninguno implicado, la consecuencia práctica es la misma:

> **Conclusión** La primera señal _muchas llamadas entrantes perdidas_ no es medible a nivel de operador individual con estos datos. Las 926 llamadas perdidas sí atribuidas, repartidas entre 239 operadores, son una base demasiado escasa.

Construir un ranking sobre esa métrica produciría un resultado que parece riguroso y no lo es. La decisión metodológica, desarrollada en la sección 4, es **separar el análisis en dos niveles**: la pérdida de llamadas entrantes se analiza como un problema de la organización cliente (cobertura y dimensionamiento), y la evaluación individual del operador se sostiene sobre las señales que sí son atribuibles.

Conviene notar además la magnitud del fenómeno: **más de la mitad de todas las llamadas entrantes se pierden**. Ese dato, por sí solo, es el hallazgo de negocio más relevante del caso.

In [ ]:
sin_dup.groupby(['direction', 'internal'], dropna=False).size().rename('filas').reset_index()

,direction,internal,filas
0,in,False,19224
1,in,True,671
2,in,NaN,108
3,out,False,24015
4,out,True,4982
5,out,NaN,2


---
## 4. Definición operativa de "operador ineficaz"

### 4.1 Dos niveles de análisis

A partir del diagnóstico, el análisis se estructura en dos niveles con destinatarios distintos:

| Nivel | Métrica | Qué diagnostica | A quién interpela |
|---|---|---|---|
| **Cliente (organización)** | Tasa de pérdida en llamadas entrantes | Cobertura y dimensionamiento del equipo | Dirección del cliente y CallMeMaybe |
| **Operador individual** | Espera en entrantes atendidas + volumen de salientes | Desempeño personal | Supervisor del equipo |



### 4.2 Métricas por operador

| Dimensión del enunciado | Métrica | Cálculo |
|---|---|---|
| Tiempo de espera prolongado | **Espera media por llamada entrante atendida** | `Σ(total_call_duration - call_duration) / Σ calls_count` en dirección `in` |
| Pocas llamadas salientes | **Volumen de llamadas salientes** | `Σ calls_count` en dirección `out` |
| Llamadas entrantes perdidas | **Tasa de pérdida atribuida** | Se calcula y reporta, pero **no pondera** en la clasificación, por la limitación de la sección 3.3 |

**Nota**: La espera se normaliza por llamada: sin normalizar, la métrica premiaría a los operadores con poca actividad.

### 4.3 Segmentación previa por perfil de actividad

El enunciado condiciona la tercera señal: *"si se supone que un operador debe realizar llamadas salientes"*. Los datos no incluyen el rol asignado, así que **el perfil se infiere del comportamiento observado** a partir de la proporción de llamadas salientes sobre el total:

- **Entrante**: hasta un 20% de salientes.
- **Mixto**: entre el 20% y el 80%.
- **Saliente**: más del 80%.

Los cortes se validarán contra la distribución real, que se espera marcadamente bimodal. Penalizar por "pocas salientes" a un operador cuyo trabajo es solo atender llamadas entrantes marcaría como ineficaz a alguien que cumple exactamente su función; por eso **el criterio de salientes solo se aplica a los perfiles mixto y saliente**, y los umbrales se calculan **dentro de cada segmento**, comparando a cada operador con sus pares.

### 4.4 Umbrales y regla de clasificación

Se usarán **percentiles dentro de cada segmento** en lugar de valores absolutos, porque no existe un estándar externo de "cuánta espera es demasiada" y porque la pregunta del supervisor es comparativa: *¿quién está peor que el resto de mi equipo?*

El percentil de corte se decidirá **mostrando la distribución de cada métrica** y justificando la elección según cuántos operadores señala: un corte que marca al 40% del equipo no es accionable, y uno que marca al 2% no justifica la función.

Un operador se clasificará como ineficaz al superar el umbral en **más de una dimensión**, evitando señalar a alguien por una única métrica que puede tener explicaciones ajenas a su desempeño.

### 4.5 Criterio de actividad mínima

Se excluirán del ranking los operadores con muy poca actividad: con muy pocas llamadas, una espera alta no es evidencia de nada. El umbral se fijará observando la distribución de actividad, verificando que los excluidos representen una fracción despreciable del volumen total.

---
## 5. Preguntas de investigación

**Sobre los datos**

1. ¿Qué calidad tienen los datos y qué limitaciones imponen al análisis?
2. ¿Cuál es el período cubierto y qué clientes y operadores hay realmente activos?

**Sobre el comportamiento del servicio**

3. ¿Cómo se distribuyen las llamadas entre entrantes/salientes e internas/externas?
4. ¿Cómo se distribuye el tiempo de espera entre operadores?
5. ¿Existen patrones temporales relevantes a lo largo del período?

**Sobre la pérdida de llamadas (nivel cliente)**

6. ¿Qué magnitud tiene la pérdida de llamadas entrantes y cómo se reparte entre clientes?
7. ¿Se relaciona con el plan tarifario contratado?

**Sobre la ineficacia (nivel operador)**

8. ¿Qué perfiles de actividad existen y cómo se reparten los operadores entre ellos?
9. ¿Qué operadores resultan ineficaces bajo la definición adoptada y cuántos son?
10. ¿Qué proporción del volumen de llamadas concentran?

---
## 6. Hipótesis a contrastar

Con base en los perfiles propuestos para los operadores y los planes de los clientes, se proponen las siguientes pruebas de hipótesis con nivel de significación: **α = 0,05**. Estas hipótesis buscan responder a dudas de negocio, como la especialización de operadores y el servicio ofrecido en los planes.



### Hipótesis 1 — Especialización y tiempo de espera

> **H₀:** El tiempo medio de espera en llamadas entrantes atendidas es igual entre los tres perfiles de operador (entrante, mixto, saliente).
>
> **H₁:** Al menos un perfil presenta un tiempo de espera distinto.

*Por qué importa:* si los operadores especializados en atender llamadas entrantes tuvieran esperas menores, la recomendación sería avanzar hacia la especialización de roles. Si fueran mayores, indicaría que están saturados y que el problema es de carga, no de aptitud. El perfil se define por la proporción de llamadas salientes, sin intervención de la variable de espera.

### Hipótesis 2 — Plan tarifario y pérdida de llamadas

> **H₀:** La tasa de pérdida en llamadas entrantes es igual entre los clientes de los planes A, B y C.
>
> **H₁:** Al menos un plan presenta una tasa de pérdida distinta.

*Por qué importa:* si los clientes de un plan pierden sistemáticamente más llamadas, el problema no está en sus operadores sino en las condiciones del servicio contratado, y la recomendación de negocio cambia por completo: revisar el plan en lugar de evaluar al equipo.

### Selección de la prueba estadística

La prueba no se elige de antemano. En cada caso se verificará **normalidad** (Shapiro-Wilk, apoyado en inspección gráfica) y **homogeneidad de varianzas** (Levene), y se usará ANOVA si se cumplen los supuestos o Kruskal-Wallis si no se cumplen. Dado que los tiempos de espera suelen presentar fuerte asimetría a la derecha, es previsible la vía no paramétrica; aun así la verificación será explícita. Si el resultado es significativo, se aplicará una comparación por pares con *orrección por comparaciones múltiples para identificar qué grupos difieren.

---
## 7. Plan de limpieza y decisiones justificadas

| # | Problema | Tratamiento | Justificación |
|---|---|---|---|
| 1 | Duplicados exactos (~9% de las filas) | **Eliminar**, reportando el volumen afectado | Comprobado en 3.2: ninguna clave lógica se repite con métricas distintas, todos los grupos tienen 2 copias idénticas y la tasa es constante en el tiempo. Es un fallo de carga |
| 2 | `operator_id` nulo (~15%) | **Conservar** para el análisis a nivel de cliente; **excluir** del análisis por operador | Comprobado en 3.3: son casi exclusivamente llamadas entrantes perdidas, que por naturaleza no tienen operador. Eliminarlas ocultaría el principal problema del negocio; imputar un operador sería inventar datos |
| 3 | `internal` nulo (117 filas, ~0,2%) | **Excluir** del desglose interna/externa, manteniéndolas en los totales | Volumen marginal que no justifica imputación, pero sí dejar constancia |
| 4 | `date` con zona horaria `+03:00` | Convertir a datetime con zona horaria explícita | Ignorarla puede desplazar registros de día y distorsionar el análisis temporal |

**Transformaciones adicionales**

- Derivar `wait_time = total_call_duration - call_duration` (verificado en 3.1: sin valores negativos).
- Derivar la espera media por llamada dividiendo entre `calls_count`.
- Unir con `telecom_clients.csv` para incorporar plan tarifario y antigüedad del cliente.

**Principio general:** `data/raw/` no se modifica. Los resultados intermedios y las tablas agregadas se guardan en `data/interim/` y `data/processed/`.

---
## 8. Etapas del análisis

| Etapa | Contenido | Responde a |
|---|---|---|
| **1. Preparación** | Carga, tipado, tratamiento de los cuatro problemas de calidad, variables derivadas, unión con clientes | Preguntas 1 y 2 |
| **2. EDA general** | Composición del tráfico, volumen diario, distribución de duraciones y esperas | Preguntas 3, 4 y 5 |
| **3. Análisis por cliente** | Tasa de pérdida de entrantes por organización y por plan tarifario | Preguntas 6 y 7 |
| **4. Perfilado de operadores** | Tabla agregada por operador; clasificación por perfil de actividad | Pregunta 8 |
| **5. Identificación de ineficaces** | Distribución por segmento, elección justificada del corte, marcado y ranking | Preguntas 9 y 10 |
| **6. Pruebas de hipótesis** | Verificación de supuestos, ejecución de H1 y H2, interpretación en lenguaje de negocio | Hipótesis 1 y 2 |
| **7. Conclusiones** | Perfil del operador ineficaz, magnitud del problema y recomendaciones accionables | Objetivo general |

### Gráficos previstos

| Gráfico | Etapa | Qué debe mostrar |
|---|---|---|
| Circular entrantes/salientes e internas/externas | 2 | Composición del tráfico |
| Serie temporal de llamadas por día | 2 | Estacionalidad y cobertura del período |
| Histograma de duración y de tiempo de espera | 2 | Forma de la distribución (justifica la elección de prueba estadística) |
| Distribución de la tasa de pérdida por cliente y por plan | 3 | Magnitud y reparto del problema de cobertura |
| Distribución de la proporción de salientes | 4 | Validación empírica de los cortes de perfil |
| Métricas por segmento con el umbral marcado | 5 | Hace visible y discutible el corte elegido |
| Dispersión espera vs. volumen de salientes | 5 | Si ambas señales identifican a los mismos operadores |
| Comparación entre grupos de cada hipótesis | 6 | Soporte visual de las pruebas estadísticas |

---
## 9. Entregables y riesgos

| Entregable | Archivo | Estado |
|---|---|---|
| Descomposición de tareas | `01_decomposicion.ipynb` | Este documento |
| Implementación del análisis | `02_analisis_operadores.ipynb` | Siguiente fase |
| Dashboard | Tableau Public (`dashboard/`) | Siguiente fase |
| Presentación de conclusiones | `presentacion/` (PDF) | Siguiente fase |

### Riesgos y limitaciones declaradas

| Riesgo | Mitigación |
|---|---|
| La señal de llamadas perdidas no es atribuible al operador | Se analiza a nivel de cliente y se declara explícitamente la limitación, en lugar de construir una métrica individual sin sustento |
| Los umbrales por percentil son una decisión metodológica, no un hecho | Mostrar la distribución, justificar el corte y reportar cuántos operadores señala |
| Se desconoce el rol real asignado a cada operador | El perfil se infiere del comportamiento observado y se declara como supuesto |
| El período cubierto es de unos cuatro meses de 2019 | No extrapolar a otros períodos; declarar el alcance temporal |
| Solo 307 de 732 clientes registran actividad | El universo de análisis son los clientes activos; se declara al inicio del análisis |